In [1]:
import os

os.environ.setdefault("HIP_VISIBLE_DEVICES", "0") # Force GPU usage instead of iGPU (for ROCm configured torch)

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

In [2]:
train_df = pd.read_parquet("data/df_train_preprocessed.parquet")
test_df = pd.read_parquet("data/df_test_preprocessed.parquet")

id_cols = ["month_decision", "weekday_decision", "WEEK_NUM", "case_id"]

# Guarantee chronological row order before dropping WEEK_NUM bcs TimeSeriesSplit has this as the assumption
train_df = train_df.sort_values(
    "WEEK_NUM", kind="mergesort").reset_index(drop=True)
test_df = test_df.sort_values(
    "WEEK_NUM", kind="mergesort").reset_index(drop=True)

# Keep weeknum for our stratisfied sampling during tuning
week_num_train = train_df["WEEK_NUM"].copy()

train_df.drop(columns=id_cols, inplace=True)
test_df.drop(columns=id_cols, inplace=True)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (1088836, 202), Test shape: (437823, 202)


## 3.0 Setup

In [3]:
# Define features and target variable
X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

In [4]:
# ── Shared tuning setup (used by all three models) ──────────────────────────
# NOTE: TimeSeriesSplit relies on row order. X_train / y_train are assumed to be
# already sorted chronologically (by decision time / WEEK_NUM) upstream, so the
# folds below respect the temporal ordering of the training block.

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

N_SPLITS = 5      # TimeSeriesSplit folds used during tuning
# Optuna trials per model (raise/lower for your compute budget)
N_TRIALS = 20

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# ── Fixed class weight ──────────────────────────────────────────────────────
# Computed ONCE on the FULL training block and shared, unchanged, across all
# three models. It is NOT recomputed per fold or per trial.
#
# Documented design choice (not a bug): because weight_ratio comes from the full
# training block, the earliest TimeSeriesSplit folds during tuning are scored
# against a class ratio that is partly informed by *later* training data. We
# accept this deliberately — a single fixed, shared weight keeps the three
# models directly comparable and matches the frozen-model evaluation design.
n_positive = int((y_train == 1).sum())
n_negative = int((y_train == 0).sum())
weight_ratio = n_negative / n_positive
print(f"weight_ratio (n_negative / n_positive) = {weight_ratio:.4f}")


def gini(y_true, y_score):
    """Gini coefficient = 2 * AUC - 1"""
    return 2.0 * roc_auc_score(y_true, y_score) - 1.0


def print_trial(study, trial):
    """Optuna callback: print the Gini and params of each completed trial."""
    print(
        f"  Trial {trial.number:>3} | Gini: {trial.value:.4f} | Params: {trial.params}")


def make_tuning_subsample(X, y, week_num, frac, seed=SEED):
    """Optional stratified subsample of the training block, for TUNING ONLY.

    Stratifies jointly on (WEEK_NUM, target): groups rows by week and class,
    draws frac of the rows within each group, then restores the original row
    order (by position). Returns the data unchanged when frac is None or >= 1.
    """
    if frac is None or frac >= 1.0:
        return X, y
    rng = np.random.default_rng(seed)
    y_arr = y.to_numpy()
    week_arr = week_num.to_numpy()

    keep_parts = []
    for week in np.unique(week_arr):
        week_mask = week_arr == week
        for cls in np.unique(y_arr):
            idx = np.where(week_mask & (y_arr == cls))[0]
            n_keep = max(1, round(len(idx) * frac)) if len(idx) > 0 else 0
            if n_keep > 0:
                keep_parts.append(rng.choice(idx, size=n_keep, replace=False))
    keep = np.sort(np.concatenate(keep_parts))
    return X.iloc[keep], y.iloc[keep]


# Fraction of the training block used during tuning to reduce compute cost. The final model is trained on the full training block.
FRAC_LR = 0.2
FRAC_XGB = 0.2
FRAC_MLP = 0.2

weight_ratio (n_negative / n_positive) = 31.3885


## 3.1 Logistische regressie

In [5]:
# Optuna tuning — Logistic Regression
X_lr, y_lr = make_tuning_subsample(X_train, y_train, week_num_train, FRAC_LR)


def lr_solver(l1_ratio):
    # sklearn 1.8+: penalty is deprecated, l1_ratio drives the penalty.
    # l1_ratio == 0.0 -> pure L2  -> newton-cholesky (2nd order, fast/accurate)
    # l1_ratio == 1.0 -> pure L1  -> liblinear (coordinate descent, purpose-built
    #   for L1; far faster and more robust than saga, which struggles to converge
    #   on pure L1 because there is no L2 term to add strong convexity).
    return "newton-cholesky" if l1_ratio == 0.0 else "liblinear"


def lr_objective(trial):
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    l1_ratio = trial.suggest_categorical("l1_ratio", [0.0, 1.0])
    solver = lr_solver(l1_ratio)

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_lr):
        model = LogisticRegression(
            C=C,
            l1_ratio=l1_ratio,
            solver=solver,
            class_weight={0: 1.0, 1: weight_ratio},
            max_iter=1000,
            random_state=SEED,
        )
        model.fit(X_lr.iloc[tr_idx], y_lr.iloc[tr_idx])
        proba = model.predict_proba(X_lr.iloc[va_idx])[:, 1]
        fold_ginis.append(gini(y_lr.iloc[va_idx], proba))
    return float(np.mean(fold_ginis))


lr_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
lr_study.optimize(lr_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best LR params:", lr_study.best_params)
print(f"Best LR mean CV Gini: {lr_study.best_value:.4f}")

# Retrain the final LR on the FULL training block with the selected params.
# Mirror the objective's solver routing based on the chosen l1_ratio.
lr_best = LogisticRegression(
    **lr_study.best_params,
    solver=lr_solver(lr_study.best_params["l1_ratio"]),
    class_weight={0: 1.0, 1: weight_ratio},
    max_iter=1000,
    random_state=SEED,
)
lr_best.fit(X_train, y_train)

  Trial   0 | Gini: 0.6033 | Params: {'C': 0.017670169402947963, 'l1_ratio': 0.0}
  Trial   1 | Gini: 0.6001 | Params: {'C': 0.39079671568228835, 'l1_ratio': 0.0}
  Trial   2 | Gini: 0.6113 | Params: {'C': 0.00022310108018679258, 'l1_ratio': 0.0}
  Trial   3 | Gini: 0.5953 | Params: {'C': 1.7718847354806828, 'l1_ratio': 1.0}
  Trial   4 | Gini: 0.5992 | Params: {'C': 9.877700294007917, 'l1_ratio': 0.0}
  Trial   5 | Gini: 0.5971 | Params: {'C': 0.0012601639723276807, 'l1_ratio': 1.0}
  Trial   6 | Gini: 0.6031 | Params: {'C': 0.039054412752107935, 'l1_ratio': 1.0}
  Trial   7 | Gini: 0.5865 | Params: {'C': 0.0006870101665590031, 'l1_ratio': 1.0}
  Trial   8 | Gini: 0.6016 | Params: {'C': 0.054502936945582565, 'l1_ratio': 0.0}
  Trial   9 | Gini: 0.6008 | Params: {'C': 0.12173252504194051, 'l1_ratio': 0.0}
  Trial  10 | Gini: 0.5992 | Params: {'C': 51.704055291544954, 'l1_ratio': 0.0}
  Trial  11 | Gini: 0.6099 | Params: {'C': 0.00011099001648454433, 'l1_ratio': 0.0}
  Trial  12 | Gini:

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.0003506736327673949
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1.0, 1: 31.388482360640133}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'newton-cholesky'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 pen

## 3.2 XGBoost

In [6]:
# Optuna tuning — XGBoost
X_xgb, y_xgb = make_tuning_subsample(
    X_train, y_train, week_num_train, FRAC_XGB)


def xgb_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_xgb):
        model = XGBClassifier(
            **params,
            scale_pos_weight=weight_ratio,  # fixed, shared weight (not tuned)
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        )
        model.fit(X_xgb.iloc[tr_idx], y_xgb.iloc[tr_idx])
        proba = model.predict_proba(X_xgb.iloc[va_idx])[:, 1]
        fold_ginis.append(gini(y_xgb.iloc[va_idx], proba))
    return float(np.mean(fold_ginis))


xgb_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best XGB params:", xgb_study.best_params)
print(f"Best XGB mean CV Gini: {xgb_study.best_value:.4f}")

# Retrain the final XGBoost on the FULL training block with the selected params.
xgb_best = XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=weight_ratio,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)
xgb_best.fit(X_train, y_train)

  Trial   0 | Gini: 0.4663 | Params: {'max_depth': 5, 'learning_rate': 0.22648248189516848, 'n_estimators': 750, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 4, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}
  Trial   1 | Gini: 0.5747 | Params: {'max_depth': 7, 'learning_rate': 0.05675206026988748, 'n_estimators': 100, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'min_child_weight': 5, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}
  Trial   2 | Gini: 0.6102 | Params: {'max_depth': 5, 'learning_rate': 0.0199473547030745, 'n_estimators': 500, 'subsample': 0.645614570099021, 'colsample_bytree': 0.8059264473611898, 'min_child_weight': 3, 'reg_alpha': 4.258943089524393e-06, 'reg_lambda': 1.9826980964985924e-05}
  Trial   3 | Gini: 0.5352 | Params: {'max_depth': 6, 'learning_rate': 0.08810003129071789, 'n_estimators': 250, 'subsample': 0.7571172192068059, 'colsample

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.5590520349254929
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## 3.3 MLP

In [7]:
# Optuna tuning — MLP (PyTorch)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FEATURES = X_train.shape[1]
MLP_EPOCHS = 15  # fixed training budget per fit (kept small for tuning)

# Fixed candidate architectures (hidden-layer sizes) keep the search space small.
MLP_ARCHITECTURES = {
    "128": [128],
    "256-128": [256, 128],
    "128-64": [128, 64],
}


class MLP(nn.Module):
    def __init__(self, n_features, hidden_sizes, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))  # single output logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(X_tr, y_tr, hidden_sizes, lr, weight_decay, batch_size, dropout):
    torch.manual_seed(SEED)
    model = MLP(N_FEATURES, hidden_sizes, dropout).to(device)
    # Fixed, shared class weight applied through the loss.
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(weight_ratio, device=device)
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay)

    ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    # shuffle=False keeps the time order of the (already ordered) training rows.
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.train()
    for _ in range(MLP_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def predict_mlp(model, X_va):
    model.eval()
    xb = torch.tensor(X_va, dtype=torch.float32).to(device)
    return torch.sigmoid(model(xb)).cpu().numpy()


# MLP tuning runs on a stratified, time-ordered subsample (see FRAC_MLP).
X_mlp, y_mlp = make_tuning_subsample(
    X_train, y_train, week_num_train, FRAC_MLP)
X_mlp_np = X_mlp.to_numpy(dtype=np.float32)
y_mlp_np = y_mlp.to_numpy(dtype=np.float32)


def mlp_objective(trial):
    arch_key = trial.suggest_categorical(
        "architecture", list(MLP_ARCHITECTURES))
    hidden_sizes = MLP_ARCHITECTURES[arch_key]
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    fold_ginis = []
    for tr_idx, va_idx in tscv.split(X_mlp_np):
        model = train_mlp(
            X_mlp_np[tr_idx], y_mlp_np[tr_idx],
            hidden_sizes, lr, weight_decay, batch_size, dropout,
        )
        proba = predict_mlp(model, X_mlp_np[va_idx])
        fold_ginis.append(gini(y_mlp_np[va_idx], proba))
    return float(np.mean(fold_ginis))


mlp_study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
mlp_study.optimize(mlp_objective, n_trials=N_TRIALS, callbacks=[print_trial])

print("Best MLP params:", mlp_study.best_params)
print(f"Best MLP mean CV Gini: {mlp_study.best_value:.4f}")

# Retrain the final MLP on the FULL training block with the selected params.
best = mlp_study.best_params
mlp_best = train_mlp(
    X_train.to_numpy(dtype=np.float32),
    y_train.to_numpy(dtype=np.float32),
    MLP_ARCHITECTURES[best["architecture"]],
    best["learning_rate"],
    best["weight_decay"],
    best["batch_size"],
    best["dropout"],
)

  Trial   0 | Gini: 0.5056 | Params: {'architecture': '256-128', 'learning_rate': 0.0015751320499779737, 'weight_decay': 4.2079886696066345e-06, 'batch_size': 1024, 'dropout': 0.3005575058716044}
  Trial   1 | Gini: 0.5117 | Params: {'architecture': '128-64', 'learning_rate': 0.004622589001020831, 'weight_decay': 7.068974950624607e-06, 'batch_size': 1024, 'dropout': 0.2623782158161189}
  Trial   2 | Gini: 0.6067 | Params: {'architecture': '128-64', 'learning_rate': 0.00019010245319870352, 'weight_decay': 1.4742753159914662e-05, 'batch_size': 1024, 'dropout': 0.09983689107917987}
  Trial   3 | Gini: 0.5188 | Params: {'architecture': '256-128', 'learning_rate': 0.0016409286730647919, 'weight_decay': 4.809461967501575e-06, 'batch_size': 1024, 'dropout': 0.40419867405823057}
  Trial   4 | Gini: 0.5246 | Params: {'architecture': '128-64', 'learning_rate': 0.0007591104805282694, 'weight_decay': 3.0771802712506896e-06, 'batch_size': 1024, 'dropout': 0.12938999080000846}
  Trial   5 | Gini: 0.

## 3.4 Summary + save models

In [8]:
# ── Tuning summary ──────────────────────────────────────────────────────────
print(f"Fixed weight_ratio (shared across all models): {weight_ratio:.4f}")
print(f"Random seed: {SEED}\n")
print("Best hyperparameters per model")
print("  LR :", lr_study.best_params, f"(CV Gini {lr_study.best_value:.4f})")
print("  XGB:", xgb_study.best_params, f"(CV Gini {xgb_study.best_value:.4f})")
print("  MLP:", mlp_study.best_params, f"(CV Gini {mlp_study.best_value:.4f})")

Fixed weight_ratio (shared across all models): 31.3885
Random seed: 42

Best hyperparameters per model
  LR : {'C': 0.0003506736327673949, 'l1_ratio': 0.0} (CV Gini 0.6114)
  XGB: {'max_depth': 4, 'learning_rate': 0.012159090549740887, 'n_estimators': 750, 'subsample': 0.6210638168418385, 'colsample_bytree': 0.5590520349254929, 'min_child_weight': 20, 'reg_alpha': 3.200570323847896e-05, 'reg_lambda': 0.0013201986749627366} (CV Gini 0.6206)
  MLP: {'architecture': '128', 'learning_rate': 0.00018184382136929325, 'weight_decay': 0.007266943511144088, 'batch_size': 256, 'dropout': 0.07474370954165258} (CV Gini 0.6239)


In [11]:
import os
import joblib

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Logistic Regression — plain pickle via joblib
lr_path = os.path.join(MODELS_DIR, "lr_best.joblib")
joblib.dump(lr_best, lr_path)

# XGBoost — native format (portable across xgboost/sklearn versions)
xgb_path = os.path.join(MODELS_DIR, "xgb_best.json")
xgb_best.save_model(xgb_path)

# MLP — state dict + the architecture metadata needed to reconstruct the model
mlp_path = os.path.join(MODELS_DIR, "mlp_best.pt")
mlp_arch = MLP_ARCHITECTURES[mlp_study.best_params["architecture"]]
torch.save(
    {
        "state_dict": mlp_best.state_dict(),
        "n_features": N_FEATURES,
        "hidden_sizes": mlp_arch,
        "dropout": mlp_study.best_params["dropout"],
    },
    mlp_path,
)

print(f"Saved LR  -> {lr_path}")
print(f"Saved XGB -> {xgb_path}")
print(f"Saved MLP -> {mlp_path}")

Saved LR  -> models/lr_best.joblib
Saved XGB -> models/xgb_best.json
Saved MLP -> models/mlp_best.pt
